# Day 3 실습 — Pydantic·Structured Output

**목표**: Pydantic 틀과 `with_structured_output`으로 구조화 출력을 받고, 중첩·리스트·검증까지 실전 스키마를 설계한다.
**구성**: Part 1 기본 구조화·오류 다루기 → Part 2 중첩·리스트·검증(+다른 도메인) → Part 3 리뷰 추출기(+이력서 자동 추출 실전 프로젝트)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키를 확인한다.

## 0. 환경 준비

In [6]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List
from enum import Enum
from dotenv import load_dotenv

# TODO
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
llm.model_name

'gpt-4o-mini'

## Part 1. 기본 구조화 출력

완성 코드를 직접 쳐서 Pydantic 틀로 구조화 답을 받는다.

> **참고:** `BaseModel`·`Field`·`Optional` 문법 자체는 FastAPI 과정(`14_fastapi`)에서 이미 써봤다. 여기서는 같은 문법으로 **LLM 응답을 구조화**한다는 점이 다르다(가이드 문서 §3-1 참고).

### 1-1. 기본 모델 → 구조화 출력

줄글이 아니라 Person 객체로 받아 `.name`으로 꺼낸다.

In [2]:
# TODO: name(str), age(int), city(str)를 Field(description=...)로 정의하세요
class Person(BaseModel):
    name: str = Field(description="사람 이름")
    age: int = Field(description="나이")
    city: str = Field(description="사는 도시")

In [3]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-openai': '1.3.4'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000019358F43890>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000193598B2810>, root_client=<openai.OpenAI object 

In [ ]:
structured_llm = llm.with_structured_output(Person) # OpenAI 에서 제공하는 기능, 랭체인x
structured_llm

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12', 'langchain-openai': '1.3.4'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000019358F43890>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000193598B2810>, root_clien

In [ ]:
result = structured_llm.invoke("김철수는 33살이고 부산에 살고 있습니다.") #응답생성에 반영
print(result)
print("이름:", result.name, "| 나이:", result.age, "| 도시:", result.city)

name='김철수' age=33 city='부산'
이름: 김철수 | 나이: 33 | 도시: 부산


In [7]:
result1 = structured_llm.invoke("홍.길.동는 스물두살이고 항구도시에 살고 있습니다.")
print("이름:", result1.name, "| 나이:", result1.age, "| 도시:", result1.city)

이름: 홍길동 | 나이: 22 | 도시: 항구도시


In [8]:
result1 = structured_llm.invoke("항구도시, 김개똥, 19")
print("이름:", result1.name, "| 나이:", result1.age, "| 도시:", result1.city)

이름: 김개똥 | 나이: 19 | 도시: 항구도시


### 1-1-보충. PydanticOutputParser와 비교

같은 결과를 예전 방식(`PydanticOutputParser`)으로 만들어 본다. 형식 지시를 프롬프트에 직접 넣고, 모델이 낸 텍스트를 파서가 사후에 파싱하는 방식이다.

In [10]:
from langchain_core.output_parsers import PydanticOutputParser

In [11]:
# TODO: PydanticOutputParser를 만들고, Person 스키마를 연결하세요
parser = PydanticOutputParser(pydantic_object=Person)

# TODO: get_format_instructions()로 얻은 형식 지시문을 프롬프트 앞에 붙이세요
prompt = f'{parser.get_format_instructions()}\n홍길동은 33살이고 항구도시에 삽니다.'

In [12]:
prompt

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "사람 이름", "title": "Name", "type": "string"}, "age": {"description": "나이", "title": "Age", "type": "integer"}, "city": {"description": "사는 도시", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```\n홍길동은 33살이고 항구도시에 삽니다.'

In [14]:
raw = llm.invoke(prompt)
print("원본 응답:", raw.content)

원본 응답: ```json
{
  "name": "홍길동",
  "age": 33,
  "city": "항구도시"
}
```


In [16]:
# TODO: parser.parse()로 raw.content를 파싱하세요
parsed = parser.parse(raw.content)
print("파싱 결과:", parsed)

파싱 결과: name='홍길동' age=33 city='항구도시'


> **참고:** 모델이 ` ```json ` 코드펜스로 감싸 답해도 `parser.parse()`가 펜스를 벗겨내고 파싱한다. `with_structured_output`(1-1)과 최종 결과는 같지만, 형식 지시를 프롬프트에 직접 넣어야 하고 파싱이 별도 단계로 분리된다는 점이 다르다(가이드 문서 §3-2 참고).

### 1-2. Optional 필드

있을 수도 없을 수도 있는 항목은 Optional로 둔다.

In [18]:
result

Person(name='김철수', age=33, city='서울')

In [19]:
result = structured_llm.invoke("김철수는 33살입니다.") #응답생성에 반영
print(result)

name='김철수' age=33 city=''


In [22]:
from typing import Optional
from enum import Enum

class Contact(BaseModel):
    name: str = Field(description="이름") # 필수
    email: Optional[str] = Field(default=None, description="이메일, 없으면 None") # 선택

In [23]:
# TODO: 있을 수도 없을 수도 있는 email을 Optional로 정의하세요

c = llm.with_structured_output(Contact)

print(c.invoke("나는 홍길동이다"))     # email 없음
print(c.invoke("홍길동이랑 연락하려면 hong@example.com으로 메일을 보내세요"))     # email 있음

name='홍길동' email='None'
name='홍길동' email='hong@example.com'


### 1-3. Enum — 값 후보 제한

감정처럼 정해진 후보 중 하나로만 받고 싶을 때 Enum을 쓴다.

In [26]:
# TODO: 긍정/부정/중립 세 후보를 정의하세요 (값이 서로 같으면 Enum 멤버가 하나로 합쳐지니 서로 다른 문자열로 채우세요)

class Sentiment(str, Enum): #범주형 컬럼/필드
    positive = "긍정"
    negative = "부정"
    neutral = "중립"

# Review 클래스
sentiment - 긍정, 부정, 중립 Sentiment 객체
reason - 평가이유 문자열

In [35]:
# TODO: sentiment(Sentiment 타입, 리뷰 감정)와 reason(str, 판단 이유)을
#       Field(description=...)로 정의하세요
class Review(BaseModel):
    sentiment: Sentiment = Field(description="리뷰 감정")
    reason: str = Field(description="평가 이유")

r = llm.with_structured_output(Review)

In [36]:
# TODO: 긍정 메시지, 부정 메시지 생성하세요
print(r.invoke("세상은 아름답다"))

sentiment=<Sentiment.positive: '긍정'> reason='자연의 아름다움과 사람들의 따뜻한 마음이 느껴지는 문장입니다.'


In [37]:
print(r.invoke("쏘쏘"))

sentiment=<Sentiment.neutral: '중립'> reason='특별한 긍정이나 부정의 감정이 드러나지 않으며, 자신이 느낀 경험을 간단히 언급한 것으로 판단됨.'


### 1-4. 오류 다뤄보기 — 정보가 없는 문장을 넣으면?

`Person`의 `name`은 필수 필드다. 이름이 아예 언급되지 않은 문장을 넣으면 어떻게 되는지 관찰한다.

In [ ]:

# TODO: 이름이 언급되지 않은 문장을 넣어보세요
result = structured_llm.invoke("333살이고 여성이고 도시에 거주합니다.") #응답생성에 반영
print(result)


name='연우' age=333 city='서울'


In [ ]:
# TODO: 예외처리를 추가하세요

> **참고:** 실행해 보면 모델이 오류를 내는 대신 이름을 지어내거나(예: "이름 없음", "미상") 빈 문자열을 채우는 경우가 많다. **구조화 출력도 환각(hallucination)을 막아주지는 않는다** — 필드가 채워졌다고 해서 그 값이 진짜라는 보장은 없다.

### 1-5. 오류 다뤄보기 — 범위를 벗어난 값을 직접 만들면?

이번엔 모델 호출이 아니라, Pydantic 객체를 코드에서 직접 만들어 **검증(Validation)**이 실제로 막는지 확인한다.

In [87]:
class PersonSafe(BaseModel):
    name: str = Field(default='null', description="이름. 텍스트에 명시적을 이름이 언급되지 않으면, 반드시 'null'로 남겨두세요"
                                                "절대 추측하거나 지어내지 마세요"
                      )
    age:int = Field(description="나이.")
    city: str = Field(description="사는 도시")

In [88]:
safe_llm = llm.with_structured_output(PersonSafe)
for _ in range(3):
    print(safe_llm.invoke("23살이고 부산에 살고 있습니다."))

name='null' age=23 city='부산'
name='null' age=23 city='부산'
name='null' age=23 city='부산'


In [89]:
safe_llm = llm.with_structured_output(PersonSafe)
for _ in range(3):
    print(safe_llm.invoke("나는 300년 묵은 구렁이다."))

name='구렁이' age=300 city='산속'
name='구렁이' age=300 city='산속'
name='구렁이' age=300 city='신비의 숲'


In [102]:
# TODO: 나이의 범위를 0~120으로 제한하세요
class PersonAgeStrict(BaseModel):
    name: str = Field(default='null', description="이름. 텍스트에 명시적을 이름이 언급되지 않으면, 반드시 'null'로 남겨두세요"
                                                "절대 추측하거나 지어내지 마세요"
                      )
    age: Optional[int] = Field(ge=0, le=120, default=0, description="나이.")
    city: str = Field(description="사는 도시")

In [103]:
safe_llm = llm.with_structured_output(PersonAgeStrict)
for _ in range(3):
    print(safe_llm.invoke("나는 세종대왕이다."))

name='세종대왕' age=None city='한양'
name='세종대왕' age=None city='한성'
name='세종대왕' age=54 city='이화'


In [104]:
# TODO: 범위를 벗어난 나이(예: 200)를 넣어보세요
age_safe_llm = llm.with_structured_output(PersonAgeStrict)
for _ in range(3):
    print(age_safe_llm.invoke("부산에 살고 있는 200세 여성입니다."))

name='null' age=None city='부산'
name='null' age=None city='부산'
name='null' age=None city='부산'


In [ ]:
# 결과에서 오류를 직접 검증해야 한다.
extracted = age_safe_llm.invoke("부산에 살고 있는 200세 여성입니다.")

#파이덴틱 모델의 유효성 검증
PersonAgeStrict.model_validate(extracted.model_dump())

PersonAgeStrict(name='null', age=None, city='부산')

In [101]:
extracted

PersonAgeStrict(name='null', age=None, city='부산')

Pydantic으로 범위를 벗어난 값을 지정하고 오류처리하기 위해서는

LLM이 아니라 직접 값을 추출해야 한다. (gpt-4o-mini 기준)

In [ ]:
try:
    extracted
except ValidationError as e:
    print(type(e).__name__)
    print(str(e)[:150])

In [105]:
try:
    PersonAgeStrict(name='null', age=200, city='부산')
except ValidationError as e:
    print(type(e).__name__)
    print(str(e)[:150])

ValidationError
1 validation error for PersonAgeStrict
age
  Input should be less than or equal to 120 [type=less_than_equal, input_value=200, input_type=int]
    For


## Part 2. 중첩·리스트·검증

단일 필드를 넘어 복잡한 구조를 표현하고, 값의 범위를 검증한다.

### 2-1. 중첩 모델 — 모델 안에 모델

주소처럼 하위 구조는 별도 모델로 표현한다.

사용자와 주소의 카디널리티 : 1대다

In [8]:
# TODO: Address를 선언하세요
class Address(BaseModel):
    city: str = Field(description="도시")
    street: str = Field(description="도로명 주소")

# TODO: Address를 하위 모델(중첩)로 넣으세요
class User(BaseModel):
    name: str = Field(description="이름")
    address: Address = Field(description="주소 정보")

In [9]:
# TODO: User를 구조화 출력으로 연결하고, "김철수, 서울시 강남대로 123번지"를 넣어 실행하세요
u = llm.with_structured_output(User)

result = u.invoke("김철수, 서울시 강남대로 123번지")
result

User(name='김철수', address=Address(city='서울시', street='강남대로 123번지'))

In [10]:
print("도시:", result.address.city)

도시: 서울시


In [12]:
print("주소:",  result.address.street)
print("유저이름:", result.name)

주소: 강남대로 123번지
유저이름: 김철수


### 2-2. 리스트 반환 — 여러 개 추출

항목이 여러 개면 `List[...]`로 받는다.

In [15]:
# TODO: Product를 정의하세요
class Product(BaseModel):
    name: str = Field(description="상품명")
    price: int = Field(description="상품 가격(원)")

# TODO: Order 를 정의하고, Product 여러 개를 담는 리스트로 정의하세요
class Order(BaseModel):
    products: List[Product] = Field(description="주문상품 목록")

In [16]:
# TODO: Order를 구조화 출력으로 연결하세요
llm_order = llm.with_structured_output(Order)

# TODO : 사과 3000원, 바나나 2000원, 우유 1500원 주문합니다 처럼 목록메시지를 작성합니다.
result = llm_order.invoke("사과 3000원, 바나나 2000원, 우유 1500원 주문합니다")
result

Order(products=[Product(name='사과', price=3000), Product(name='바나나', price=2000), Product(name='우유', price=1500)])

In [21]:
for n in result.products:
    print(str(type(n)) + n.name + str(n.price))

<class '__main__.Product'>사과3000
<class '__main__.Product'>바나나2000
<class '__main__.Product'>우유1500


### 2-3. 필드 검증·설명

Field에 범위나 설명을 달면 값이 안정된다. `ge=1, le=5`로 1~5 범위를 벗어난 값을 막는다.

In [23]:
# TODO: Rating을 정의하세요
# TODO: 1~5 범위 제한(ge, le)과 설명을 넣으세요
class Rating(BaseModel):
    score: int = Field(description="1~5 사이 평점", ge=1, le=5)

In [26]:
# TODO: Rating을 구조화 출력으로 연결하세요
llm_rating = llm.with_structured_output(Rating)

# 정말 최고입니다.
# 아쉬워요 더 보고 싶어요.
# 배송은 빨랐는데, 포장이 부실해서 찌그러져서 왔어요
# 실제 써보니 가격대비 성능은 만족스러웠지만, 설명서가 부실해서 좀 헤맷어요

In [36]:
review_list = [
    "정말 최고입니다",
    "아쉬워요 더 보고 싶어요",
    "배송은 빨랐는데, 포장이 부실해서 찌그러져서 왔어요",
    "실제 써보니 가격대비 성능은 만족스러웠지만, 설명서가 부실해서 좀 헤맷어요",
    "우유가 터졌어요. 최악이에요. 1점이 아까워요"
]

for text in review_list:
    result_rating = llm_rating.invoke(text)
    print(text + " : " + str(result_rating.score))

정말 최고입니다 : 5
아쉬워요 더 보고 싶어요 : 4
배송은 빨랐는데, 포장이 부실해서 찌그러져서 왔어요 : 3
실제 써보니 가격대비 성능은 만족스러웠지만, 설명서가 부실해서 좀 헤맷어요 : 4
우유가 터졌어요. 최악이에요. 1점이 아까워요 : 1


## Part 2-확장. 다른 도메인에 적용하기 — 회의록 액션 아이템 추출

중첩·리스트·Enum을 한꺼번에 써서, 회의록에서 "누가 무엇을 언제까지 얼마나 중요하게" 해야 하는지 뽑아본다. Day02의 회의 시간 잡기 퍼즐과 같은 소재를 이어서 쓴다.

### 2-4. 스키마 설계 — 액션 아이템

In [49]:
# TODO: Priority 를 정의하세요
# TODO: 높음/보통/낮음 세 후보를 정의하세요 (값이 서로 같으면 Enum 멤버가 하나로 합쳐지니 서로 다른 문자열로 채우세요)
class Priority(str, Enum):
    high = "높음"
    normal = "보통"
    low = "낮음"

# TODO: ActionItem를 정의하세요
class ActionItem(BaseModel):
    task: str = Field(description="해야 할 일")
    owner: str = Field(description="담당자 이름")
    # TODO: Priority를 타입으로 쓰세요
    priority: Priority = Field(description="중요도/우선순위")


# TODO: MeetingSummary를 정의하세요
class MeetingSummary(BaseModel):
    topic: str = Field(description="회의 주제")

    # TODO: ActionItem 여러 개를 담는 리스트로 정의하세요
    action_items: List[ActionItem] = Field(description="액션 아이템의 목록")


### 2-5. 회의록에 적용

In [50]:
meeting_note = (
    "오늘 회의 주제는 신규 가입 페이지 개선이다. "
    "철수가 이번 주까지 회원가입 폼 UI를 수정하기로 했고, 급한 건이다. "
    "영희는 다음 주까지 이메일 인증 로직을 점검하기로 했다. 급하지는 않다."
)

In [51]:
# TODO: MeetingSummary를 구조화 출력으로 연결하세요
meeting_ext_llm = llm.with_structured_output(MeetingSummary)

In [52]:
# TODO: meeting_note를 넣어 추출을 실행하세요
meeting_ext_result = meeting_ext_llm.invoke(meeting_note)

In [53]:
print(meeting_ext_result.topic)
print(meeting_ext_result.action_items)

신규 가입 페이지 개선
[ActionItem(task='회원가입 폼 UI 수정', owner='철수', priority=<Priority.high: '높음'>), ActionItem(task='이메일 인증 로직 점검', owner='영희', priority=<Priority.normal: '보통'>)]


In [54]:
first_item = meeting_ext_result.action_items[0]


In [58]:
print(f'담당자 : {first_item.owner}')
print(f'우선순위 : {first_item.priority.value}')

담당자 : 철수
우선순위 : 높음


In [63]:
for item in  meeting_ext_result.action_items:
    print(f'담당자 : {item.owner}  | 우선순위 : {item.priority.value} | 태스크 : {item.task}')

담당자 : 철수  | 우선순위 : 높음 | 태스크 : 회원가입 폼 UI 수정
담당자 : 영희  | 우선순위 : 보통 | 태스크 : 이메일 인증 로직 점검


### 관찰 정리

- 회의록 도메인에서도 중첩(ActionItem 안의 정보)·리스트(여러 액션 아이템)·Enum(우선순위)이 그대로 통했는가?
- 이 스키마를 실제 업무에 쓴다면, 어떤 필드를 더 추가하고 싶은가? (예: 마감일)

## Part 3. 미니 프로젝트 — 리뷰 → 구조화 JSON 추출기

오늘 배운 요소(필드·리스트·검증)를 한 과제에 모아 실전 추출기를 완성한다.

### 3-1. 추출 스키마 설계

In [64]:
# TODO: ReviewInfo를 정의하세요
# TODO: 장점·단점을 문자열 리스트로, 요약을 문자열로 정의하세요
class ReviewInfo(BaseModel):
    rating: int = Field(description="1~5 평점", ge=1, le=5)
    pros: List[str] = Field(description="장점 목록")
    cons: List[str] = Field(description="단점 목록")
    summary: str = Field(description="한 줄 요약")

In [65]:
# TODO: ReviewInfo를 구조화 출력으로 연결하세요
review_info_llm = llm.with_structured_output(ReviewInfo)

### 3-2. 여러 리뷰에 적용

In [66]:
reviews = [
    "배송은 빨랐지만 포장이 부실했어요. 그래도 품질은 만족합니다.",
    "가격 대비 최고! 디자인도 예쁘고 튼튼해요. 다만 색이 화면과 조금 달라요.",
]

In [71]:
# TODO: reviews를 넣어 추출을 실행하세요
for review in reviews:
    result = review_info_llm.invoke(reviews)
    print(f'별점 : {result.rating}')
    print(f'리뷰 한 줄 요약 : {result.summary}')
    print(f'장점 : {result.pros}')
    print(f'단점 : {result.cons}')

별점 : 4
리뷰 한 줄 요약 : 가격 대비 훌륭한 제품이지만 색상에 유의해야 합니다.
장점 : ['가격 대비 품질이 뛰어나다.', '디자인이 예쁘고 세련되다.', '내구성이 좋고 튼튼하다.']
단점 : ['색상이 화면에서 본 것과 다소 다르다.']
별점 : 4
리뷰 한 줄 요약 : 전반적으로 만족스러운 제품입니다.
장점 : ['가격 대비 뛰어난 품질', '예쁜 디자인', '튼튼한 내구성']
단점 : ['색상이 화면과 다르게 보임']


## Part 3-확장. 실전 프로젝트 — 이력서에서 지원자 정보 자동 추출

Day01·Day02에서는 `candidate_info`를 손으로 직접 입력했다. 오늘은 이력서 텍스트에서 **자동으로** 뽑아, 그대로 면접 코치 프롬프트에 연결한다.

### 지원자 정보 스키마 설계

In [ ]:
# TODO: CandidateInfo를 정의하세요

# TODO: 기술 스택은 여러 개이므로 문자열 리스트로 정의하세요

# TODO: CandidateInfo를 구조화 출력으로 연결하세요


### 이력서 텍스트에서 추출

In [ ]:
resume_text = (
    "안녕하세요, 백엔드 개발에 관심 있는 신입 지원자입니다. "
    "학교에서 Python을 배웠고, 3개월간 스타트업에서 인턴으로 FastAPI와 PostgreSQL을 다뤄봤습니다. "
    "꼼꼼하게 문서를 남기는 습관이 있습니다."
)

# TODO: resume_text를 넣어 추출을 실행하세요


### 면접 코치에 바로 연결하기

Day01에서 손으로 썼던 `candidate_info` 문자열을, 방금 추출한 결과로 대신 만든다.

In [ ]:
# TODO: candidate.career, candidate.skills를 이용해 지원자 이력 문자열을 만드세요


# TODO: ChatPromptTemplate를 이용해 면접 질문을 만드는 프롬프트를 정의하세요

# TODO: candidate_info_auto를 출력하세요


> **참고:** 이 셀은 `ChatPromptTemplate`·`StrOutputParser`를 함께 쓴다. Day01에서 만든 체인과 구조가 같다 — 오늘 배운 구조화 출력이 그 체인의 **입력을 자동으로 준비하는 단계**로 연결된 것이다.

### 정리

**확인 질문**
- 손으로 입력했던 정보와 자동 추출한 정보는 결과에 차이가 있었는가?
- 이력서 텍스트가 애매하거나 정보가 부족하면 어떤 필드가 가장 불안정해질 것 같은가?

## 확인 문제

1. 줄글 답과 구조화 답은 값을 꺼낼 때 무엇이 다른가?
2. Optional과 Enum은 각각 언제 쓰는가?
3. 1-4에서 확인한 것처럼, 텍스트에 없는 정보를 필수 필드로 요구하면 어떤 일이 생기는가?
4. 1-5에서 확인한 것처럼, `ge`·`le` 범위를 벗어난 값을 넣으면 왜 오류가 나는가?
5. 중첩 모델은 언제 쓰는가? 하위 값에는 어떻게 접근하는가?
6. Part 2-확장에서, 회의록이라는 다른 도메인에서도 중첩·리스트·Enum은 그대로 통했는가?
7. Part 3-확장에서 구조화 출력으로 뽑은 결과를, 왜 굳이 다시 프롬프트에 넣어 체인으로 연결했는가?